In [39]:
import pandas as pd
users = pd.read_csv(r'C:\Users\днс\Downloads\users.csv', encoding='koi8-r', sep="\t")
users.columns = ["user_id", "email", "geo"]
users.user_id = users.user_id.apply(lambda x: x.lower())

log = pd.read_csv(r'C:\Users\днс\Downloads\log.csv', header=None)
log.columns = ["user_id", "time", "bet", "win"]
log = log[log.user_id != "#error"]
log.user_id = log.user_id.str.split(" - ").apply(lambda x: x[1])
log.time = log.time.str.strip('[')
log['time'] = pd.to_datetime(log['time'])
log[['bet', 'win']]=log[['bet', 'win']].fillna(0)
log['net']=log['win']-log['bet']
log['month'] = log['time'].dt.month
min_common_month = log['month'].value_counts().idxmin()
week = log['time'].dt.weekday
week.value_counts()
weekend = week.value_counts().loc[6] + week.value_counts().loc[5]

def time_of_day(time):
    if 0 <= time <= 5:
        return 'ночь'
    elif 6 <= time <= 11:
        return 'утро'
    elif 12 <= time <= 17:
        return 'день'
    elif 18 <= time <= 23:
        return 'вечер'
    
log['daytime'] = log['time'].dt.hour.apply(time_of_day)

def fillna_win(row):
    if row['win'] > 0:
        return row['win']
    elif row['bet'] > 0:
        return -row.bet
    else:
        return 0 
    
log['win'] = log.apply(lambda row: fillna_win(row), axis=1)
lost = len(log[log.win<0])
print(f"Участники проиграли {lost} раз")

positive_wins_count = (log['net'] > 0).sum()
print(f"Положительный выигрыш был {positive_wins_count} раз")

positive_net = log[log['net'] > 0]
mean_value = positive_net['net'].mean()
median_value = positive_net['net'].median()
print(f"Среднее значение выигрыша: {int(mean_value)}")
print(f"Медианное значение выигрыша: {int(median_value)}")

successful_visits = (log['bet'] > 0).sum()
total_visits = log['bet'].count()
visit_rate = successful_visits / total_visits
print("Процент посещений букмекерской конторы, который заканчивается ставкой: {:.2%}".format(visit_rate))

successful_bet = log[log['bet'] > 0]
mean_bet = successful_bet['bet'].mean()
print(f"Среднее значение ставки: {int(mean_bet)}")

log_users = pd.merge(users, log, on='user_id')  
median_users = log_users.groupby('user_id').net.sum().median()
print(f"Медиана баланса по каждому пользователю: {int(median_users)}")

user_id_with_bets = list(log_users[log_users.bet > 0].groupby('user_id').user_id.count().index.values)
put_bet_data = log_users[log_users.user_id.isin(user_id_with_bets)]
bet_user = log_users[log_users.user_id.isin(user_id_with_bets) & (log_users.bet == 0)].groupby('user_id').user_id.count().mean()
print(f"В среднем приходит, не делая ставок, каждый человек, у которого была хотя бы одна ставка {bet_user} раз")

first_visit_df = put_bet_data[put_bet_data.bet == 0].groupby('user_id').time.min().reset_index()
first_visit_df = first_visit_df.rename(columns={'time': 'first_visit_time'})
first_bet_df = put_bet_data[put_bet_data.bet > 0].groupby('user_id').time.min().reset_index()
first_bet_df = first_bet_df.rename(columns={'time': 'first_bet_time'})
new_df = pd.merge(first_visit_df, first_bet_df, on='user_id')
new_df['timedelta'] = new_df['first_bet_time'] - new_df['first_visit_time']
result = new_df[new_df['timedelta'].dt.days >= 0].timedelta.mean()
print(f"В среднем проходит между появлением человека в сервисе и первой ставкой: {result.round('s')}")




Участники проиграли 339 раз
Положительный выигрыш был 133 раз
Среднее значение выигрыша: 82625
Медианное значение выигрыша: 5329
Процент посещений букмекерской конторы, который заканчивается ставкой: 47.92%
Среднее значение ставки: 6806
Медиана баланса по каждому пользователю: 1986
В среднем приходит, не делая ставок, каждый человек, у которого была хотя бы одна ставка 5.05 раз
В среднем проходит между появлением человека в сервисе и первой ставкой: 49 days 13:01:39


Можно сделать вывод, что есть несколько крупных выигрышей, которые выводят среднее значение вверх. Может показаться, что типичный выигрыш гораздо больше, чем есть на самом деле. Медианное значение же показывает, что типичный выигрыш составляет 5347, что ближе к реальности у большинства случаев.

Высокая доля «холостых» визитов (не ставящих пользователей). Процент посещений, заканчивающихся ставкой, составляет всего 47.92 %. Это значит, что почти каждый второй визит не конвертируется в ставку. Вероятно, часть аудитории приходит ради просмотра линии, коэффициентов или статистики, а не для игры. Это важно учитывать при оценке реальной монетизации трафика и при расчёте CAC (стоимости привлечения клиента).

Низкая частота ставок у активных пользователей. Даже среди тех, кто делает ставки, в среднем на одного такого пользователя приходится лишь 5.05 ставок (при учёте всех визитов). Это указывает на эпизодический характер игры у большинства: пользователи не являются регулярными игроками, а заходят время от времени. Отсюда следует, что удержание и повторные конверсии — ключевая зона роста.